### Diffusion of a gaussian packegue

Number density of each flavor component along $x$. Each curve is one snapshot; color is time.

In [ ]:
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import h5py
import numpy as np

try:
    here = Path(__file__).resolve().parent
except NameError:
    here = Path.cwd()

sys.path.append(str((here / '..' / 'nsm_plots').resolve()))
sys.path.append(str((here / '..' / 'phys_const').resolve()))

import plot_functions as pf
import phys_const as pc

Gaussian diffusion test simulation paramters

In [ ]:
# Directory path
directory_path = '/home/erick/software/Emu/Exec'
number_of_flavors = 3

# Simulation domain
ncell = (1024, 1, 1)
Lx = 200.0 # cm
Ly = 0.1953125 # cm
Lz = 0.1953125  # cm
domain = (Lx, Ly, Lz)  # cm

# Scattering opacities
kappar_cm = 10.0
kapparbar_cm = 10.0

# Initial conditions of Gaussian packet
# n = n_bg + A0 * exp(-(x - x0)^2 / (2 s0^2))
n_bg_ccm = 1.0e28
A0_ccm = 1.0e29
x0_cm = 100.0
s0_cm = 5.0

# Snapshots drawn on each spatial plot
lowlimfit = None
uplimfit = None

Analitic solution

In [ ]:
D_cm2_s = pc.PhysConst.c / (3.0 * kappar_cm)
dx = domain[0] / ncell[0]
D_num_cm2_s = pc.PhysConst.c * kappar_cm * dx**2 / 4.0
D_cm2_s += D_num_cm2_s

def analytic_solution(t_s, x_cm=None):
    """Cartesian reference density n(x, t) in cm^-3.

    n(x, t) = n_bg + A0 * s0 / sqrt(s0^2 + 2 D t)
              * exp(-(x - x0)^2 / (2 (s0^2 + 2 D t)))

    t_s is a time in seconds, or an array of times. x_cm defaults to the
    cell centers of the run. A scalar time returns shape (nx,); an array
    of times returns shape (nt, nx).
    """
    if x_cm is None:
        dx = domain[0] / ncell[0]
        x_cm = np.arange(dx / 2.0, domain[0], dx)
    x_cm = np.asarray(x_cm, dtype=float)
    t_in = np.asarray(t_s, dtype=float)
    variance = s0_cm**2 + 2.0 * D_cm2_s * np.atleast_1d(t_in)
    n = (
        n_bg_ccm
        + A0_ccm * s0_cm / np.sqrt(variance)[:, None]
        * np.exp(-((x_cm - x0_cm) ** 2) / (2.0 * variance[:, None]))
    )
    if t_in.ndim == 0:
        return n[0]
    return n

Reading simulation data

In [ ]:
all_data_path = os.path.join(directory_path, 'allData.h5')
with h5py.File(all_data_path, 'r') as all_data_file:
    nee = np.array(all_data_file['/N00_Re(1|ccm)'][:])
    nuu = np.array(all_data_file['/N11_Re(1|ccm)'][:])
    neu = np.sqrt(
        np.array(all_data_file['/N01_Re(1|ccm)'][:])**2
        + np.array(all_data_file['/N01_Im(1|ccm)'][:])**2
    )
    if number_of_flavors == 3:
        ntt = np.array(all_data_file['/N22_Re(1|ccm)'][:])
        net = np.sqrt(
            np.array(all_data_file['/N02_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N02_Im(1|ccm)'][:])**2
        )
        nut = np.sqrt(
            np.array(all_data_file['/N12_Re(1|ccm)'][:])**2
            + np.array(all_data_file['/N12_Im(1|ccm)'][:])**2
        )
    time_s = np.array(all_data_file['/t(s)'][:])

Plotting data

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x_cell_centers = np.arange(domain[0] / ncell[0] / 2, domain[0], domain[0] / ncell[0])

# Calculate analytic solution at initial and last time
analytic_init = analytic_solution(time_s[0], x_cell_centers)
analytic_final = analytic_solution(time_s[-1], x_cell_centers)

# Plot simulation results
ax.plot(x_cell_centers, nee[0, :, 0, 0], label=r"$n_{ee}$ (t=0)", color="C1")
ax.plot(x_cell_centers, nee[-1, :, 0, 0], label=rf"$n_{{ee}}$ (t={time_s[-1]:.2e} s)", color="red")

# Plot analytic solution (black dotted line)
ax.plot(x_cell_centers, analytic_init, "k:", label="analytic (t=0)")
ax.plot(x_cell_centers, analytic_final, "k--", label=f"analytic (t={time_s[-1]:.2e} s)")

ax.set_xlabel(r"$x$ [cm]")
ax.set_ylabel(r"$n_{ee}$ [cm$^{-3}$]")
ax.legend(fontsize=ax.legend().get_texts()[0].get_fontsize() / 2)
plt.tight_layout()
plt.show()